# LSTM / GRU — Bayesian Context Features

Same four recurrent models as `lstm_gru_comparison.ipynb` but **Level-5 context
features swap from MAUT gap scores → Bayesian θ posteriors** from the hierarchical model.

| Old L5 context (MAUT) | New L5 context (Bayesian) |
|---|---|
| gap_score_balanced | theta_median |
| gap_score_cerf/echo/usaid/ngo | theta_ci_lo, theta_ci_hi, theta_ci_width |
| median_rank, rank_iqr | completeness |

**Reference (MAUT features):**
| Model | MAE | UFE@10 | UFE@15 |
|---|---|---|---|
| Mamba2 | 0.1224 | 8/10 | 10/15 |
| GRU-Bi | 0.1282 | 8/10 | 12/15 |
| GRU-Uni | 0.1411 | 6/10 | 10/15 |
| LSTM-Uni | 0.1431 | 5/10 | 9/15 |
| LSTM-Bi | 0.1431 | 7/10 | 10/15 |

### Upload these files when prompted:
1. `monthly_severity_sequences.npy`  — from `Data/`
2. `sequence_countries.csv`          — from `Data/`
3. `sequence_snapshots.csv`          — from `Data/`
4. `enriched_frame_2025.csv`         — from `Data/`
5. `cerf_ufe.csv`                    — from `Data/Third-Party/Benchmarks/`
6. `care_bts.csv`                    — from `Data/Third-Party/Benchmarks/`
7. `analysis_code.zip`               — from `bayesian_features/`
8. `bayesian_theta_cache.parquet`    — **optional** (from `mamba_bayesian.ipynb` run, skips ~90s recomputation)

In [ ]:
# Install deps
!pip install numpyro jax einops -q

In [ ]:
from google.colab import files
import zipfile, os

print('Upload files: monthly_severity_sequences.npy, sequence_countries.csv,')
print('sequence_snapshots.csv, enriched_frame_2025.csv, cerf_ufe.csv, care_bts.csv,')
print('analysis_code.zip  [+ optional: bayesian_theta_cache.parquet]')

uploaded = files.upload()
for fname, data in uploaded.items():
    with open(f'/content/{fname}', 'wb') as f:
        f.write(data)
    print(f'Saved {fname} ({len(data)/1024:.1f} KB)')

# Unzip analysis code
if os.path.exists('/content/analysis_code.zip'):
    with zipfile.ZipFile('/content/analysis_code.zip', 'r') as z:
        z.extractall('/content/')
    print('Extracted analysis_code.zip → /content/analysis/')

In [ ]:
import warnings, sys, time
warnings.filterwarnings('ignore')
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/analysis')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DATA   = Path('/content')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42)
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Compute Bayesian θ posteriors (~60-90s on CPU)

Fits the hierarchical latent-variable model (SVI, AutoMultivariateNormal, 8000 steps)
on 6 MAUT attributes from the enriched frame. Outputs per-country:
- `theta_median` — posterior median (higher = more overlooked)
- `theta_ci_lo/hi` — 90% credible interval
- `theta_ci_width` — CI width (uncertainty proxy)
- `completeness` — share of attributes observed

If you uploaded `bayesian_theta_cache.parquet` this step is skipped.

In [ ]:
enriched_raw = pd.read_csv(DATA / 'enriched_frame_2025.csv', index_col=0)

THETA_CACHE = DATA / 'bayesian_theta_cache.parquet'
if THETA_CACHE.exists():
    print('Loading cached Bayesian posteriors...')
    theta_df = pd.read_parquet(THETA_CACHE)
else:
    print('Fitting Bayesian hierarchical model...')
    from aggregations.composites import compute_overlookedness_posterior
    theta_df = compute_overlookedness_posterior(enriched_raw)
    theta_df.to_parquet(THETA_CACHE)
    print('Cached to bayesian_theta_cache.parquet')

enriched = enriched_raw.copy()
for col in ['theta_median', 'theta_ci_lo', 'theta_ci_hi', 'theta_ci_width', 'completeness']:
    if col in theta_df.columns:
        enriched[col] = theta_df[col]

hrp = enriched['theta_median'].notna()
print(f'HRP-eligible countries with theta: {hrp.sum()} / {len(enriched)}')
print(enriched.loc[hrp, ['theta_median','theta_ci_lo','theta_ci_hi','theta_ci_width']].describe().round(3))

## 2. Data pipeline — same sequences, new context

In [ ]:
X_monthly = np.load(DATA / 'monthly_severity_sequences.npy')   # (75, 65, 4)
countries  = pd.read_csv(DATA / 'sequence_countries.csv',  header=None)[0].tolist()
snapshots  = pd.read_csv(DATA / 'sequence_snapshots.csv',  header=None)[0].tolist()
cerf_ufe   = pd.read_csv(DATA / 'cerf_ufe.csv')
care_bts   = pd.read_csv(DATA / 'care_bts.csv')
N, T, F_seq = X_monthly.shape

# Context: L2 structural + L3 concentration + L4 temporal + L5 Bayesian
CTX_COLS = [
    'need_intensity', 'coverage_shortfall', 'coverage', 'per_pin_allocated',  # L2
    'cluster_gini', 'donor_hhi', 'cbpf_reliance',                             # L3
    'severity_baseline_24m', 'severity_acute_delta_3m',                       # L4
    'severity_volatility_12m', 'severity_trend_12m',
    'coverage_baseline_3y', 'coverage_trend_3y',
    'displaced_growth_12m', 'persistence_P4_plus',
    'theta_median', 'theta_ci_lo', 'theta_ci_hi', 'theta_ci_width',           # L5 Bayesian
    'completeness',
]
CTX_COLS = [c for c in CTX_COLS if c in enriched.columns]
F_ctx    = len(CTX_COLS)
print(f'Context features: {F_ctx}  (MAUT version had 21)')
print(f'  Bayesian L5: {[c for c in CTX_COLS if "theta" in c or c=="completeness"]}')

ctx_frame = enriched.reindex(countries)[CTX_COLS].values.astype(np.float64)
ctx_med   = np.nanmedian(ctx_frame, axis=0)
for j in range(F_ctx):
    ctx_frame[:, j] = np.where(np.isnan(ctx_frame[:, j]), ctx_med[j], ctx_frame[:, j])
ctx_scaler = StandardScaler().fit(ctx_frame)
ctx_norm   = ctx_scaler.transform(ctx_frame).astype(np.float32)

seq_flat   = X_monthly[:, :48, :].reshape(-1, F_seq)
seq_scaler = StandardScaler().fit(seq_flat)
X_norm     = seq_scaler.transform(X_monthly.reshape(-1, F_seq)).reshape(N, T, F_seq).astype(np.float32)

SEQ_LEN, PRED_STEPS = 24, 3
seqs_X, seqs_ctx, seqs_y_sev, seqs_meta = [], [], [], []
for i, iso3 in enumerate(countries):
    for t in range(SEQ_LEN, T - PRED_STEPS):
        seqs_X.append(X_norm[i, t-SEQ_LEN:t])
        seqs_ctx.append(ctx_norm[i])
        seqs_y_sev.append(float(np.mean(X_monthly[i, t:t+PRED_STEPS, 0])))
        seqs_meta.append({'iso3': iso3, 'end_snapshot': snapshots[t]})

X_arr   = np.stack(seqs_X).astype(np.float32)
ctx_mat = np.stack(seqs_ctx).astype(np.float32)
y_arr   = np.array(seqs_y_sev, dtype=np.float32)
meta_df = pd.DataFrame(seqs_meta)
train_mask = meta_df['end_snapshot'] < '2025-01'
train_idx  = meta_df[train_mask].index.tolist()
test_idx   = meta_df[~train_mask].index.tolist()

neglect_map = {}
for iso3 in countries:
    if iso3 in enriched.index:
        cov = enriched.loc[iso3, 'coverage']
        th  = enriched.loc[iso3, 'theta_median'] if 'theta_median' in enriched.columns else np.nan
        neglect_map[iso3] = {
            'neglected': float(not pd.isna(cov) and cov < 0.4),
            'gap_score': float(th) if not pd.isna(th) else 0.,
        }

y_neglect = np.array([neglect_map.get(m['iso3'], {}).get('neglected', 0.5)
                      for m in seqs_meta], dtype=np.float32)
y_gap     = np.array([neglect_map.get(m['iso3'], {}).get('gap_score', 0.)
                      for m in seqs_meta], dtype=np.float32)
if y_gap.std() > 0:
    y_gap = (y_gap - y_gap.min()) / (y_gap.max() - y_gap.min())

n_pos      = float(y_neglect[train_idx].sum())
n_neg      = float(len(train_idx)) - n_pos
pos_weight = torch.tensor([n_neg / max(n_pos, 1.)], dtype=torch.float32, device=DEVICE)

print(f'Train: {len(train_idx)}  Test: {len(test_idx)}  Target: {y_arr.min():.1f}–{y_arr.max():.1f}')

## 3. Model definitions — same architecture as lstm_gru_comparison.ipynb

In [ ]:
class MultiTargetDataset(Dataset):
    def __init__(self, X, ctx, y_sev, y_gap, y_neg):
        self.X     = torch.tensor(X,     dtype=torch.float32)
        self.ctx   = torch.tensor(ctx,   dtype=torch.float32)
        self.y_sev = torch.tensor(y_sev, dtype=torch.float32)
        self.y_gap = torch.tensor(y_gap, dtype=torch.float32)
        self.y_neg = torch.tensor(y_neg, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.ctx[i], self.y_sev[i], self.y_gap[i], self.y_neg[i]

def make_loaders():
    tr = MultiTargetDataset(X_arr[train_idx], ctx_mat[train_idx],
                            y_arr[train_idx], y_gap[train_idx], y_neglect[train_idx])
    te = MultiTargetDataset(X_arr[test_idx],  ctx_mat[test_idx],
                            y_arr[test_idx],  y_gap[test_idx],  y_neglect[test_idx])
    return DataLoader(tr, 64, shuffle=True), DataLoader(te, 128, shuffle=False)

train_loader, test_loader = make_loaders()


class RNNForecaster(nn.Module):
    """Context-conditioned LSTM/GRU with bidirectional support."""
    def __init__(self, F_seq, F_ctx, rnn_type='lstm', bidirectional=False,
                 hidden=48, n_layers=2, dropout=0.15):
        super().__init__()
        D       = 2 if bidirectional else 1
        rnn_out = hidden * D
        pool    = rnn_out * 2
        self.input_proj = nn.Linear(F_seq, hidden)
        self.ctx_proj   = nn.Sequential(nn.Linear(F_ctx, hidden), nn.LayerNorm(hidden), nn.GELU())
        rnn_cls = nn.LSTM if rnn_type == 'lstm' else nn.GRU
        self.rnn = rnn_cls(hidden, hidden, n_layers, batch_first=True,
                           dropout=dropout if n_layers > 1 else 0.,
                           bidirectional=bidirectional)
        self.norm    = nn.LayerNorm(rnn_out)
        self.dropout = nn.Dropout(dropout)
        def head(): return nn.Sequential(nn.Linear(pool, 32), nn.GELU(), nn.Linear(32, 1))
        self.sev_head = head(); self.gap_head = head(); self.neglect_head = head()

    def forward(self, x, ctx):
        h = self.input_proj(x) + self.ctx_proj(ctx).unsqueeze(1)  # broadcast context
        h = self.dropout(h)
        out, _ = self.rnn(h)
        out = self.norm(out)
        pool = torch.cat([out[:, -1, :], out.mean(1)], -1)
        return (self.sev_head(pool).squeeze(-1),
                self.gap_head(pool).squeeze(-1),
                self.neglect_head(pool).squeeze(-1))


configs = {
    'LSTM-Uni': dict(rnn_type='lstm', bidirectional=False),
    'LSTM-Bi':  dict(rnn_type='lstm', bidirectional=True),
    'GRU-Uni':  dict(rnn_type='gru',  bidirectional=False),
    'GRU-Bi':   dict(rnn_type='gru',  bidirectional=True),
}

def count_params(m): return sum(p.numel() for p in m.parameters())

print('Parameter counts with Bayesian context (F_ctx may differ from MAUT baseline):')
for name, cfg in configs.items():
    m = RNNForecaster(F_seq, F_ctx, hidden=48, n_layers=2, dropout=0.15, **cfg)
    print(f'  {name:12s}: {count_params(m):>8,}')

## 4. Training — same config as lstm_gru_comparison.ipynb

In [ ]:
EPOCHS, LR, PATIENCE = 150, 8e-4, 30
W_SEV, W_GAP, W_NEG  = 0.5, 0.3, 0.2

def train_model(model, name):
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=3e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, epochs=EPOCHS,
        steps_per_epoch=len(train_loader), pct_start=0.15)
    best, state, pat = float('inf'), None, 0
    tr_l, te_l, te_m = [], [], []
    t0 = time.time()

    for epoch in range(EPOCHS):
        model.train(); ep = 0.
        for X_b, ctx_b, y_sev_b, y_gap_b, y_neg_b in train_loader:
            X_b, ctx_b = X_b.to(DEVICE), ctx_b.to(DEVICE)
            y_sev_b, y_gap_b, y_neg_b = y_sev_b.to(DEVICE), y_gap_b.to(DEVICE), y_neg_b.to(DEVICE)
            sp, gp, nl = model(X_b, ctx_b)
            loss = (W_SEV * F.huber_loss(sp, y_sev_b, delta=0.5) +
                    W_GAP * F.huber_loss(gp, y_gap_b, delta=0.5) +
                    W_NEG * F.binary_cross_entropy_with_logits(nl, y_neg_b, pos_weight=pos_weight))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            opt.step(); sched.step(); ep += loss.item()

        model.eval(); tl = 0.; pred_, true_ = [], []
        with torch.no_grad():
            for X_b, ctx_b, y_sev_b, y_gap_b, y_neg_b in test_loader:
                X_b, ctx_b = X_b.to(DEVICE), ctx_b.to(DEVICE)
                y_sev_b, y_gap_b, y_neg_b = y_sev_b.to(DEVICE), y_gap_b.to(DEVICE), y_neg_b.to(DEVICE)
                sp, gp, nl = model(X_b, ctx_b)
                tl += (W_SEV * F.huber_loss(sp, y_sev_b, delta=0.5) +
                       W_GAP * F.huber_loss(gp, y_gap_b, delta=0.5) +
                       W_NEG * F.binary_cross_entropy_with_logits(nl, y_neg_b, pos_weight=pos_weight)).item()
                pred_.extend(sp.cpu().numpy()); true_.extend(y_sev_b.cpu().numpy())

        tl /= len(test_loader)
        mae = mean_absolute_error(true_, pred_)
        tr_l.append(ep / len(train_loader)); te_l.append(tl); te_m.append(mae)

        if tl < best: best, state, pat = tl, {k:v.clone() for k,v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= PATIENCE: print(f'  [{name}] early stop ep={epoch}'); break

        if epoch % 20 == 0:
            print(f'  [{name}] ep={epoch:3d} | test={tl:.4f} | MAE={mae:.4f}')

    model.load_state_dict(state)
    print(f'  [{name}] best={best:.4f}  ({time.time()-t0:.0f}s)')
    return model, best, tr_l, te_l, te_m

In [ ]:
results = {}
for name, cfg in configs.items():
    print(f'\n=== {name} ===')
    m = RNNForecaster(F_seq, F_ctx, hidden=48, n_layers=2, dropout=0.15, **cfg)
    trained, best, tr_l, te_l, te_m = train_model(m, name)
    results[name] = dict(model=trained, best_loss=best, train_losses=tr_l,
                         test_losses=te_l, test_maes=te_m, params=count_params(trained))

## 5. Evaluation

In [ ]:
def evaluate(model):
    model.eval(); pred_, true_ = [], []
    with torch.no_grad():
        for X_b, ctx_b, y_sev_b, *_ in test_loader:
            sp, _, _ = model(X_b.to(DEVICE), ctx_b.to(DEVICE))
            pred_.extend(sp.cpu().numpy()); true_.extend(y_sev_b.numpy())
    pred = np.array(pred_); true = np.array(true_)
    return dict(MAE=mean_absolute_error(true, pred),
                RMSE=np.sqrt(mean_squared_error(true, pred)),
                Naive_MAE=mean_absolute_error(true, np.full_like(true, true.mean())),
                pred=pred)

eval_results = {name: evaluate(res['model']) for name, res in results.items()}

print('MAE / RMSE comparison (lower = better):')
print(f'{"Model":<12} {"Params":>8}  {"MAE":>7}  {"RMSE":>7}  {"vs Naive":>9}  {"BestLoss":>9}')
print('-'*60)
# MAUT baselines (hardcoded reference)
print(f'{"[MAUT]Mamba2":12} {110739:>8,}  {0.1224:>7.4f}  {0.2223:>7.4f}  {6.59:>8.2f}×  {"—":>9}')
print(f'{"[MAUT]GRU-Bi":12} {90483:>8,}  {0.1282:>7.4f}  {0.2275:>7.4f}  {6.29:>8.2f}×  {"—":>9}')
print()
for name, res in results.items():
    ev = eval_results[name]
    print(f'{name:<12} {res["params"]:>8,}  {ev["MAE"]:>7.4f}  {ev["RMSE"]:>7.4f}  '
          f'{ev["Naive_MAE"]/ev["MAE"]:>8.2f}×  {res["best_loss"]:>9.4f}')

In [ ]:
def safe_norm(s):
    mn, mx = s.min(), s.max()
    return (s-mn)/(mx-mn) if mx > mn else pd.Series(0., index=s.index)

def build_ranking(model):
    model.eval(); rows = []
    for i, iso3 in enumerate(countries):
        xt = torch.tensor(X_norm[i, -SEQ_LEN:][None], dtype=torch.float32).to(DEVICE)
        ct = torch.tensor(ctx_norm[i][None],           dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            sp, _, nl = model(xt, ct)
        cur = float(X_monthly[i, -1, 0])
        row = {'iso3': iso3, 'current_severity': cur,
               'predicted_severity_3m': float(sp.cpu()),
               'neglect_prob': float(torch.sigmoid(nl).cpu())}
        if iso3 in enriched.index:
            for col in ['theta_median', 'severity_baseline_24m', 'theta_ci_width']:
                row[col] = enriched.loc[iso3, col] if col in enriched.columns else np.nan
        row['severity_worsening'] = max(0., row['predicted_severity_3m'] - cur)
        rows.append(row)
    df = pd.DataFrame(rows)
    df['theta_norm']      = safe_norm(df['theta_median'].fillna(0))
    df['sev_norm']        = safe_norm(df['severity_worsening'].fillna(0))
    df['neglect_norm']    = safe_norm(df['neglect_prob'].fillna(0))
    df['composite_score'] = 0.40*df['theta_norm'] + 0.35*df['sev_norm'] + 0.25*df['neglect_norm']
    mask = df['severity_baseline_24m'].fillna(0) >= 3
    r = df[mask].sort_values('composite_score', ascending=False).reset_index(drop=True)
    r['rank'] = r.index + 1
    return r

ufe_set = set(cerf_ufe['iso3'])
bts_set = set(care_bts['iso3'])

print('CERF UFE overlap — Bayesian context models:')
print(f'{"Model":<12}  @5    @10    @15  | CARE@10')
print('-' * 48)
for name, res in results.items():
    ranking = build_ranking(res['model'])
    results[name]['ranking'] = ranking
    ov  = {k: len(set(ranking['iso3'].head(k)) & ufe_set) for k in [5, 10, 15]}
    bts = len(set(ranking['iso3'].head(10)) & bts_set)
    print(f'{name:<12}  {ov[5]}/5   {ov[10]}/10   {ov[15]}/15  | {bts}/10')

In [ ]:
rows = []
# MAUT baselines (hardcoded from local training runs)
maut_ref = [
    ('Mamba2',  'MAUT', 110739, 0.1224, 0.2223, 3, 8, 10),
    ('GRU-Bi',  'MAUT',  90483, 0.1282, 0.2275, 3, 8, 12),
    ('GRU-Uni', 'MAUT',  39123, 0.1411, 0.2304, 3, 6, 10),
    ('LSTM-Bi', 'MAUT', 113907, 0.1431, 0.2383, 2, 7, 10),
    ('LSTM-Uni','MAUT',  48531, 0.1431, 0.2359, 2, 5, 9),
]
for name, ctx, params, mae, rmse, u5, u10, u15 in maut_ref:
    rows.append(dict(Model=f'[MAUT] {name}', Context=ctx, Params=params,
                     MAE=mae, RMSE=rmse, UFE5=u5, UFE10=u10, UFE15=u15))

# Bayesian results
for name, res in results.items():
    ev  = eval_results[name]
    rk  = res['ranking']
    ov  = {k: len(set(rk['iso3'].head(k)) & ufe_set) for k in [5, 10, 15]}
    rows.append(dict(Model=f'[Bayesian] {name}', Context='Bayesian θ',
                     Params=res['params'], MAE=ev['MAE'], RMSE=ev['RMSE'],
                     UFE5=ov[5], UFE10=ov[10], UFE15=ov[15]))

compare = pd.DataFrame(rows).set_index('Model')
print('\n=== FULL COMPARISON: MAUT vs Bayesian context ===')
print(compare.to_string())
compare.to_csv('/content/model_comparison_bayesian_rnn.csv')
print('\nSaved: model_comparison_bayesian_rnn.csv')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']

for ax, (name, res), color in zip(axes, results.items(), colors):
    ax.plot(res['train_losses'], color=color, alpha=0.5, lw=2, label='Train')
    ax.plot(res['test_losses'],  color=color,            lw=2, label='Test')
    ax2 = ax.twinx()
    ax2.plot(res['test_maes'], 'gray', lw=1.5, ls='--', label='MAE')
    ax2.axhline(0.1224, color='red', ls=':', lw=1, label='MAUT Mamba2')
    ax2.set_ylabel('MAE', fontsize=9)
    best_mae = min(res['test_maes'])
    ax.set_title(f'[Bayesian ctx] {name}  MAE={best_mae:.4f}', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    l1, lb1 = ax.get_legend_handles_labels()
    l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lb1+lb2, fontsize=8)

plt.suptitle('LSTM/GRU with Bayesian θ context features', y=1.01)
plt.tight_layout()
plt.savefig('/content/lstm_gru_bayesian_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for name, res in results.items():
    fname = '/content/' + name.lower().replace('-','_') + '_bayesian_model.pt'
    torch.save(res['model'].state_dict(), fname)
    print(f'Saved: {fname}')

# Download outputs
files.download('/content/model_comparison_bayesian_rnn.csv')
files.download('/content/lstm_gru_bayesian_curves.png')
# Download theta cache if it wasn't uploaded (so future runs skip recomputation)
if not (DATA / 'bayesian_theta_cache.parquet').stat().st_size == 0:
    files.download('/content/bayesian_theta_cache.parquet')